# 02: Extracting Embeddings from NotePlan Files

This notebook demonstrates how to generate vector embeddings from NotePlan notes for semantic search.

## Prerequisites

**⚠️ Important:** Before running this notebook, ensure you have:
- Completed [**00-import.ipynb**](./00-import.ipynb) for environment detection and LiteLLM proxy setup

All environment detection, LiteLLM proxy configuration, and NotePlan directory setup are handled in `00-import.ipynb`.

## Overview

This notebook generates vector embeddings from your NotePlan notes. Embeddings are mathematical representations that capture the semantic meaning of text, allowing you to search by meaning rather than exact keywords. This notebook focuses on generating the embeddings - you'll load them into a vector store (Neo4j or Qdrant) in subsequent notebooks.

We'll:
1. Load NotePlan files
2. Generate embeddings using LiteLLM proxy
3. Display embedding statistics and sample embeddings

**Note:** This notebook generates embeddings but doesn't store them. Use [**04o1-loading-vector-embeddings-neo4j.ipynb**](./04o1-loading-vector-embeddings-neo4j.ipynb) or [**04o2-loading-vector-embeddings-qdrant.ipynb**](./04o2-loading-vector-embeddings-qdrant.ipynb) to store them in a vector database.


In [ ]:
# Run common imports and setup
%run 00-import.ipynb

# Additional imports specific to this notebook
from knowledge_agents.utils.graph_utils import read_noteplan_files_with_metadata
from knowledge_agents.utils.vector_store_utils import generate_embeddings

# NotePlan utilities
from notes.traversal import get_files_from_last_month

print("✅ Additional libraries imported")

## Load NotePlan Files

Get NotePlan files to generate embeddings for.

In [ ]:
# Get NotePlan files from the last month
files = get_files_from_last_month(NOTEPLAN_DIR)
print(f"Found {len(files)} files to process")

# Limit to first 10 files for demo (remove this limit for full processing)
files = files[:10]
print(f"Processing {len(files)} files for this demo")
# Note: read_noteplan_files_with_metadata() will automatically filter out files that should be skipped

In [ ]:
# Read file contents using utility function
file_contents, file_metadata = read_noteplan_files_with_metadata(
    files=files,
    noteplan_dir=NOTEPLAN_DIR,
    include_file_path_in_content=False,  # Just use content, don't prepend file path
    skip_database_files=True,
)

print(f"✅ Read {len(file_contents)} files")
print(f"   Sample file: {file_metadata[0]['file_path'] if file_metadata else 'N/A'}")

## Generate Embeddings

Generate embeddings using LiteLLM proxy. These embeddings capture the semantic meaning of your notes.

In [ ]:
# Generate embeddings using LiteLLM proxy
print("Generating embeddings...")
print(f"   Model: {settings.litellm_proxy_embedding_model}")
print(f"   Proxy: http://{settings.litellm_proxy_host}:{settings.litellm_proxy_port}")

embeddings = generate_embeddings(
    texts=file_contents,
    dependencies=dependencies,
    batch_size=10,
    embedding_model=settings.litellm_proxy_embedding_model,
)

print(f"\n✅ Generated {len(embeddings)} embeddings")
print(f"   Embedding dimension: {len(embeddings[0]) if embeddings else 0}")
print(f"   Total vectors: {len(embeddings)}")

## Inspect Embeddings

Examine the generated embeddings to understand their structure.

In [ ]:
# Display embedding statistics
if embeddings:
    import numpy as np
    
    # Convert to numpy array for analysis
    embedding_array = np.array(embeddings)
    
    print("Embedding Statistics:")
    print(f"   Shape: {embedding_array.shape}")
    print(f"   Mean: {embedding_array.mean():.4f}")
    print(f"   Std: {embedding_array.std():.4f}")
    print(f"   Min: {embedding_array.min():.4f}")
    print(f"   Max: {embedding_array.max():.4f}")
    
    # Show sample embedding (first few dimensions)
    print(f"\nSample embedding (first 10 dimensions):")
    print(embeddings[0][:10])
    
    # Show which file this embedding corresponds to
    if file_metadata:
        print(f"\nCorresponds to file: {file_metadata[0]['file_path']}")

## Next Steps

Now that embeddings are generated, proceed to:
- [**04o1-loading-vector-embeddings-neo4j.ipynb**](./04o1-loading-vector-embeddings-neo4j.ipynb): Store embeddings in Neo4j vector store
- [**04o2-loading-vector-embeddings-qdrant.ipynb**](./04o2-loading-vector-embeddings-qdrant.ipynb): Store embeddings in Qdrant vector store

**Note:** The variables `embeddings`, `file_contents`, and `file_metadata` are available in this notebook's scope. If you want to use them in another notebook, you'll need to either:
1. Run this notebook first, then run the loading notebook in the same kernel session
2. Save the embeddings to a file and load them in the next notebook
3. Re-generate embeddings in the loading notebook (which is the recommended approach for reproducibility)